# Human Resources Candidate Ranking
## Notebook 02 — Grouped Cross-Validation and Model Selection

Fold-safe grouped evaluation; run from the repository root.

In [1]:
import hashlib
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.stem.snowball import SnowballStemmer
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

LABELLED_DATA = "potential-talents-labelled.csv"
QUERIES = ("aspiring human resources", "seeking human resources")
RELEVANT_THRESHOLD = 2.0
STEMMER = SnowballStemmer("english")
HR_TERMS = {"human", "resource", "resources", "hr", "hris"}
INTENT_TERMS = {"aspiring", "seeking", "student", "internship", "entry", "entrylevel"}

def surface_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_text(text):
    text = surface_text(text)
    text = re.sub(r"\bhr\b", " human resources ", text)
    text = re.sub(r"\bentry level\b", " entrylevel ", text)
    return re.sub(r"\s+", " ", text).strip()

def preprocess(text, mode):
    if mode == "surface":
        return surface_text(text)
    normalized = normalize_text(text)
    if mode == "normalized":
        return normalized
    if mode == "porter2":
        return " ".join(STEMMER.stem(token) for token in normalized.split())
    raise ValueError(f"Unknown preprocessing mode: {mode}")

def parse_connections(value):
    match = re.search(r"\d+", str(value).replace(",", ""))
    return float(match.group()) if match else np.nan

def stable_group_key(job_title, location, connection_num):
    payload = "|".join([
        normalize_text(job_title),
        normalize_text(location),
        "" if pd.isna(connection_num) else f"{float(connection_num):g}",
    ])
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

df = pd.read_csv(LABELLED_DATA)
assert len(df) == 104
assert df["id"].is_unique
assert df["final_revised_grade"].notna().all()
assert set(df["final_revised_grade"].astype(int).unique()).issubset({0, 1, 2, 3})

df["connection_num"] = df["connection"].map(parse_connections)
df["group_key"] = [
    stable_group_key(t, l, c)
    for t, l, c in zip(df["job_title"], df["location"], df["connection_num"])
]
assert df.groupby("group_key")["final_revised_grade"].nunique().max() == 1

groups = (
    df.sort_values("id")
      .groupby("group_key", sort=False)
      .agg(
          canonical_id=("id", "min"),
          member_ids=("id", lambda s: tuple(int(x) for x in s)),
          duplicate_count=("id", "size"),
          job_title=("job_title", "first"),
          location=("location", "first"),
          connection_num=("connection_num", "first"),
          relevance=("final_revised_grade", "first"),
      )
      .reset_index()
      .sort_values("canonical_id")
      .reset_index(drop=True)
)
assert len(groups) == 53

display(pd.DataFrame({
    "quantity": ["raw rows", "unique profile groups", "relevant profiles (grade >= 2)"],
    "value": [len(df), len(groups), int((groups["relevance"] >= RELEVANT_THRESHOLD).sum())],
}))

def dcg_at_k(y_true, y_score, k):
    y_true = np.asarray(y_true, float)
    y_score = np.asarray(y_score, float)
    order = np.argsort(-y_score, kind="mergesort")[:k]
    rel = y_true[order]
    return float(np.sum((2 ** rel - 1) / np.log2(np.arange(2, len(rel) + 2)))) if len(rel) else 0.0

def ndcg_at_k(y_true, y_score, k):
    actual = dcg_at_k(y_true, y_score, k)
    ideal = dcg_at_k(y_true, y_true, k)
    return actual / ideal if ideal > 0 else 0.0

def ap_at_k(y_true, y_score, k, threshold=RELEVANT_THRESHOLD):
    y_true = np.asarray(y_true, float)
    y_score = np.asarray(y_score, float)
    relevant = y_true >= threshold
    total = int(relevant.sum())
    if total == 0:
        return 0.0
    hits = relevant[np.argsort(-y_score, kind="mergesort")[:k]]
    precisions = [hits[:i+1].mean() for i in range(len(hits)) if hits[i]]
    return float(sum(precisions) / min(total, k))

def reciprocal_rank(y_true, y_score, threshold=RELEVANT_THRESHOLD):
    y_true = np.asarray(y_true, float)
    order = np.argsort(-np.asarray(y_score, float), kind="mergesort")
    hits = np.flatnonzero(y_true[order] >= threshold)
    return 1.0 / (hits[0] + 1) if len(hits) else 0.0

def ranking_metrics(y_true, y_score):
    return {
        "ndcg@5": ndcg_at_k(y_true, y_score, 5),
        "ndcg@10": ndcg_at_k(y_true, y_score, 10),
        "map@5": ap_at_k(y_true, y_score, 5),
        "map@10": ap_at_k(y_true, y_score, 10),
        "mrr": reciprocal_rank(y_true, y_score),
    }

def bm25_training_statistics(train_docs):
    train_tokens = [doc.split() for doc in train_docs]
    n_train = len(train_tokens)
    avgdl = float(np.mean([len(tokens) for tokens in train_tokens]))
    doc_freq = Counter()
    for tokens in train_tokens:
        doc_freq.update(set(tokens))
    return n_train, avgdl, doc_freq

def bm25_score_documents(doc_texts, query_text, n_train, avgdl, doc_freq, k1=1.5, b=0.75):
    query_tokens = query_text.split()
    scores = []
    for doc in doc_texts:
        tokens = doc.split()
        tf = Counter(tokens)
        dl = len(tokens)
        score = 0.0
        for term in query_tokens:
            df = doc_freq.get(term, 0)
            idf = math.log(1.0 + (n_train - df + 0.5) / (df + 0.5))
            f = tf.get(term, 0)
            if f:
                norm = 1 - b + b * dl / max(avgdl, 1e-12)
                score += idf * (f * (k1 + 1)) / (f + k1 * norm)
        scores.append(score)
    return np.asarray(scores, dtype=float)

def build_fold_features(train_groups, validation_groups, mode):
    train_docs = [preprocess(x, mode) for x in train_groups["job_title"]]
    validation_docs = [preprocess(x, mode) for x in validation_groups["job_title"]]
    queries = [preprocess(q, mode) for q in QUERIES]

    word_vectorizer = TfidfVectorizer(
        tokenizer=str.split, preprocessor=None, token_pattern=None,
        ngram_range=(1, 2), sublinear_tf=True
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True
    )

    word_train = word_vectorizer.fit_transform(train_docs)
    word_validation = word_vectorizer.transform(validation_docs)
    word_queries = word_vectorizer.transform(queries)

    char_train = char_vectorizer.fit_transform(train_docs)
    char_validation = char_vectorizer.transform(validation_docs)
    char_queries = char_vectorizer.transform(queries)

    n_train, avgdl, doc_freq = bm25_training_statistics(train_docs)
    hr_set = set(preprocess(" ".join(HR_TERMS), mode).split())
    intent_set = set(preprocess(" ".join(INTENT_TERMS), mode).split())

    def make_rows(profile_groups, docs, word_matrix, char_matrix):
        rows = []
        for query_id, query in enumerate(queries):
            query_set = set(query.split())
            query_hr = query_set & hr_set
            query_intent = query_set & intent_set

            bm25_scores = bm25_score_documents(
                docs, query, n_train, avgdl, doc_freq
            )
            word_scores = cosine_similarity(
                word_matrix, word_queries[query_id]
            ).ravel()
            char_scores = cosine_similarity(
                char_matrix, char_queries[query_id]
            ).ravel()

            for i, row in profile_groups.reset_index(drop=True).iterrows():
                doc_set = set(docs[i].split())
                union = query_set | doc_set
                rows.append({
                    "query_id": query_id,
                    "query_text": QUERIES[query_id],
                    "group_key": row["group_key"],
                    "canonical_id": int(row["canonical_id"]),
                    "relevance": float(row["relevance"]),
                    "bm25": float(bm25_scores[i]),
                    "tfidf_word": float(word_scores[i]),
                    "tfidf_char": float(char_scores[i]),
                    "jaccard": len(query_set & doc_set) / len(union) if union else 0.0,
                    "containment": len(query_set & doc_set) / len(query_set) if query_set else 0.0,
                    "hr_overlap": len(query_hr & doc_set) / len(query_hr) if query_hr else 0.0,
                    "intent_overlap": len(query_intent & doc_set) / len(query_intent) if query_intent else 0.0,
                    "log_connections": (
                        float(np.log1p(row["connection_num"]))
                        if pd.notna(row["connection_num"]) else 0.0
                    ),
                })
        return pd.DataFrame(rows)

    return (
        make_rows(train_groups, train_docs, word_train, char_train),
        make_rows(validation_groups, validation_docs, word_validation, char_validation),
    )

PREPROCESSING_MODES = ["surface", "normalized", "porter2"]
REPRESENTATIONS = {
    "tfidf_word": ["tfidf_word"],
    "tfidf_word_char": ["tfidf_word", "tfidf_char"],
    "bm25": ["bm25"],
    "hybrid": ["bm25", "tfidf_word", "tfidf_char"],
}
FEATURE_BUNDLES = {
    "retrieval": [],
    "retrieval_overlap": ["jaccard", "containment"],
    "retrieval_overlap_intent": ["jaccard", "containment", "hr_overlap", "intent_overlap"],
    "retrieval_overlap_intent_connections": [
        "jaccard", "containment", "hr_overlap", "intent_overlap", "log_connections"
    ],
}
ALPHAS = [0.1, 1.0, 10.0, 100.0]

splitter = GroupKFold(n_splits=5)
fold_indices = list(splitter.split(groups, groups["relevance"], groups["group_key"]))

fold_feature_cache = {}
for fold, (train_index, validation_index) in enumerate(fold_indices, 1):
    train_groups = groups.iloc[train_index].reset_index(drop=True)
    validation_groups = groups.iloc[validation_index].reset_index(drop=True)
    assert set(train_groups["group_key"]).isdisjoint(validation_groups["group_key"])
    for mode in PREPROCESSING_MODES:
        fold_feature_cache[(fold, mode)] = build_fold_features(
            train_groups, validation_groups, mode
        )

search_rows = []
observation_rows = []

for mode in PREPROCESSING_MODES:
    for representation, retrieval_columns in REPRESENTATIONS.items():
        for feature_bundle, extras in FEATURE_BUNDLES.items():
            columns = retrieval_columns + extras
            for alpha in ALPHAS:
                observations = []
                config_name = f"{mode}|{representation}|{feature_bundle}|ridge_a{alpha:g}"

                for fold, _ in enumerate(fold_indices, 1):
                    train_features, validation_features = fold_feature_cache[(fold, mode)]
                    model = Pipeline([
                        ("scale", StandardScaler()),
                        ("ridge", Ridge(alpha=alpha)),
                    ])
                    model.fit(train_features[columns], train_features["relevance"])
                    validation = validation_features.copy()
                    validation["prediction"] = model.predict(validation[columns])

                    for query_id, query_frame in validation.groupby("query_id"):
                        metrics = ranking_metrics(
                            query_frame["relevance"], query_frame["prediction"]
                        )
                        observations.append(metrics)
                        observation_rows.append({
                            "config": config_name,
                            "fold": fold,
                            "query_id": int(query_id),
                            **metrics,
                        })

                search_rows.append({
                    "config": config_name,
                    "preprocess": mode,
                    "representation": representation,
                    "feature_bundle": feature_bundle,
                    "alpha": alpha,
                    "features": ", ".join(columns),
                    "mean_ndcg10": float(np.mean([x["ndcg@10"] for x in observations])),
                    "std_ndcg10": float(np.std([x["ndcg@10"] for x in observations], ddof=1)),
                    "mean_map10": float(np.mean([x["map@10"] for x in observations])),
                    "mean_mrr": float(np.mean([x["mrr"] for x in observations])),
                })

broad_search = (
    pd.DataFrame(search_rows)
      .sort_values(
          ["mean_ndcg10", "mean_map10", "mean_mrr"],
          ascending=[False, False, False]
      )
      .reset_index(drop=True)
)
observations = pd.DataFrame(observation_rows)

assert len(broad_search) == 192
print("Configurations evaluated:", len(broad_search))
print("Porter2 variants:", int((broad_search["preprocess"] == "porter2").sum()))
display(broad_search.head(15))

best_by_preprocessing = (
    broad_search.sort_values("mean_ndcg10", ascending=False)
    .groupby("preprocess", as_index=False).head(1)
    [["preprocess", "representation", "feature_bundle", "alpha",
      "mean_ndcg10", "std_ndcg10", "mean_map10", "mean_mrr"]]
)
best_by_representation = (
    broad_search.sort_values("mean_ndcg10", ascending=False)
    .groupby("representation", as_index=False).head(1)
    [["preprocess", "representation", "feature_bundle", "alpha",
      "mean_ndcg10", "std_ndcg10", "mean_map10", "mean_mrr"]]
)
display(best_by_preprocessing)
display(best_by_representation)

selected = broad_search.iloc[0].copy()
display(selected.to_frame("value"))

best_without_connections = broad_search[
    ~broad_search["feature_bundle"].str.contains("connections")
].iloc[0]
best_with_connections = broad_search[
    broad_search["feature_bundle"].str.contains("connections")
].iloc[0]

connection_comparison = pd.DataFrame([
    {
        "variant": "best text/retrieval model",
        "config": best_without_connections["config"],
        "mean_ndcg10": best_without_connections["mean_ndcg10"],
        "std_ndcg10": best_without_connections["std_ndcg10"],
        "mean_map10": best_without_connections["mean_map10"],
        "mean_mrr": best_without_connections["mean_mrr"],
    },
    {
        "variant": "best model including connections",
        "config": best_with_connections["config"],
        "mean_ndcg10": best_with_connections["mean_ndcg10"],
        "std_ndcg10": best_with_connections["std_ndcg10"],
        "mean_map10": best_with_connections["mean_map10"],
        "mean_mrr": best_with_connections["mean_mrr"],
    },
])
display(connection_comparison)

ax = connection_comparison.set_index("variant")["mean_ndcg10"].plot(
    kind="bar", figsize=(7, 4), title="Connection-count feature comparison"
)
ax.set_ylabel("Mean grouped-CV NDCG@10")
plt.tight_layout()
plt.show()

baseline_rows = []
for fold, _ in enumerate(fold_indices, 1):
    _, validation_features = fold_feature_cache[(fold, "normalized")]
    for query_id, query_frame in validation_features.groupby("query_id"):
        for model_name, score_column in [
            ("BM25", "bm25"),
            ("Word TF-IDF", "tfidf_word"),
            ("Char TF-IDF", "tfidf_char"),
        ]:
            metrics = ranking_metrics(
                query_frame["relevance"], query_frame[score_column]
            )
            baseline_rows.append({
                "fold": fold,
                "query_id": int(query_id),
                "model": model_name,
                **metrics,
            })

baseline_observations = pd.DataFrame(baseline_rows)
baseline_summary = (
    baseline_observations.groupby("model", as_index=False)
    .agg(
        mean_ndcg10=("ndcg@10", "mean"),
        std_ndcg10=("ndcg@10", "std"),
        mean_map10=("map@10", "mean"),
        mean_mrr=("mrr", "mean"),
    )
    .sort_values("mean_ndcg10", ascending=False)
    .reset_index(drop=True)
)
display(baseline_summary)

paired = baseline_observations.pivot_table(
    index=["fold", "query_id"], columns="model", values="ndcg@10"
)
differences = (paired["BM25"] - paired["Word TF-IDF"]).to_numpy()
mean_difference = float(np.mean(differences))
test = stats.ttest_rel(paired["BM25"], paired["Word TF-IDF"])
standard_error = stats.sem(differences)
ci_low, ci_high = stats.t.interval(
    0.95, len(differences) - 1,
    loc=mean_difference, scale=standard_error
)

paired_summary = pd.DataFrame({
    "quantity": [
        "Mean paired difference (BM25 - Word TF-IDF)",
        "Paired t-test p-value",
        "95% CI lower",
        "95% CI upper",
        "Wins",
        "Ties",
        "Losses",
        "Paired fold-query observations",
    ],
    "value": [
        mean_difference,
        float(test.pvalue),
        float(ci_low),
        float(ci_high),
        int((differences > 1e-12).sum()),
        int((np.abs(differences) <= 1e-12).sum()),
        int((differences < -1e-12).sum()),
        len(differences),
    ],
})
display(paired_summary)

checkpoint = {
    "preprocess": str(selected["preprocess"]),
    "representation": str(selected["representation"]),
    "feature_bundle": str(selected["feature_bundle"]),
    "alpha": float(selected["alpha"]),
    "mean_ndcg10": float(selected["mean_ndcg10"]),
    "std_ndcg10": float(selected["std_ndcg10"]),
    "mean_map10": float(selected["mean_map10"]),
    "mean_mrr": float(selected["mean_mrr"]),
    "best_connections_ndcg10": float(best_with_connections["mean_ndcg10"]),
    "bm25_minus_word_tfidf": float(mean_difference),
}
print("CHECKPOINT", checkpoint)

CHECKPOINT {'preprocess': 'normalized', 'representation': 'hybrid', 'feature_bundle': 'retrieval_overlap_intent', 'alpha': 10.0, 'mean_ndcg10': 0.9627884265318623, 'std_ndcg10': 0.062098363630470055, 'mean_map10': 1.0, 'mean_mrr': 1.0, 'best_connections_ndcg10': 0.9562301474072404, 'bm25_minus_word_tfidf': 0.016812728233368945}
